# Otsu Thresholding: Critical Points Visualization

This notebook explains the Otsu method by visualizing:
- **Histogram**: Distribution of pixel brightness values
- **Cumulative Distribution**: Percentage of pixels at each brightness level
- **Between-Class Variance**: Quality metric for each possible threshold
- **Optimal Threshold**: The brightness value that best separates foreground from background

The goal is to understand how Otsu automatically selects the best threshold by maximizing the variance *between* the two classes (bright/dark) while minimizing variance *within* each class.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Project imports
from src.preprocessing.preprocess_utils import make_red_enhanced, process_red_by_mode, center_crop, get_mean_brightness, classify_brightness

## 1. Load and Prepare Image Data

Load a sample red-enhanced image from the pipeline output and prepare it for histogram analysis.

In [ ]:
# Load the red-enhanced image
image_path = Path("c:/Users/issas/Desktop/new_ellipse_detection_project/data/processed/pipeline_runs/pipeline_run_101_106_v001/102_LEFT/ellipse_tester_output_fine_sweep/IMG_20260513_143554_905_roi_red.png")

if not image_path.exists():
    print(f"Image not found at {image_path}")
    print("Using alternative path...")
    # Try alternative
    image_path = Path("data/processed/pipeline_runs/pipeline_run_101_106_v001/102_LEFT/ellipse_tester_output_fine_sweep/IMG_20260513_143554_905_roi_red.png")

red_img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)

if red_img is None:
    print("Could not load image. Make sure the path is correct.")
else:
    print(f"Image shape: {red_img.shape}")
    print(f"Pixel value range: {red_img.min()} to {red_img.max()}")
    print(f"Mean brightness: {red_img.mean():.1f}")

## 2. Calculate Cumulative Pixel Distribution

Compute the histogram (count of pixels at each intensity level) and the cumulative distribution (percentage of pixels with intensity ≤ threshold).

In [ ]:
# Compute histogram
hist, bin_edges = np.histogram(red_img, bins=256, range=(0, 256))

# Normalize histogram to get probabilities
hist_norm = hist.astype(np.float32) / hist.sum()

# Compute cumulative distribution
cumsum = np.cumsum(hist_norm)

# Also compute cumulative from the other direction (percentage of pixels BRIGHTER than threshold)
cumsum_inv = 1.0 - cumsum

print("Histogram computed:")
print(f"  Total pixels: {red_img.size}")
print(f"  Bins with pixels: {(hist > 0).sum()}")
print(f"  Brightest bin: {np.argmax(hist)} (count={hist.max()})")

## 3. Compute Between-Class Variance for All Possible Thresholds

The Otsu method finds the threshold that **maximizes the between-class variance**.

For each threshold $t$:
- **Class 0** (dark): pixels with intensity $< t$ (weight $w_0 = \sum_{i=0}^{t-1} p_i$)
- **Class 1** (bright): pixels with intensity $\geq t$ (weight $w_1 = 1 - w_0$)
- **Between-class variance**: $\sigma_b^2(t) = w_0 \cdot w_1 \cdot (\mu_1 - \mu_0)^2$

We compute this for all possible thresholds and find the one with maximum variance.

In [ ]:
# Compute between-class variance for all thresholds
intensity_values = np.arange(0, 256)
global_mean = np.sum(intensity_values * hist_norm)

between_class_variance = np.zeros(256)

for t in range(256):
    w0 = cumsum[t]  # proportion of dark pixels
    w1 = 1.0 - w0   # proportion of bright pixels
    
    if w0 == 0 or w1 == 0:
        between_class_variance[t] = 0
        continue
    
    # Mean intensities of each class
    mu0 = np.sum(intensity_values[:t] * hist_norm[:t]) / w0 if w0 > 0 else 0
    mu1 = np.sum(intensity_values[t:] * hist_norm[t:]) / w1 if w1 > 0 else 0
    
    # Between-class variance
    between_class_variance[t] = w0 * w1 * (mu1 - mu0) ** 2

# Find optimal threshold
otsu_threshold = np.argmax(between_class_variance)
max_variance = between_class_variance[otsu_threshold]

print(f"Otsu optimal threshold: {otsu_threshold}")
print(f"Maximum between-class variance: {max_variance:.2f}")

# Apply Otsu with OpenCV for verification
_, otsu_cv2 = cv2.threshold(red_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
otsu_threshold_cv2 = otsu_cv2[0] if isinstance(otsu_cv2, tuple) else otsu_threshold
print(f"OpenCV Otsu threshold: {otsu_threshold}")

## 4. Visualize Critical Points: Cumulative Distribution and Between-Class Variance

This plot shows:
- **Top panel**: Cumulative distribution of pixels (% of pixels with brightness ≤ threshold)
- **Bottom panel**: Between-class variance (objective function) showing where Otsu picks the threshold

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Panel 1: Histogram
ax = axes[0]
ax.bar(range(256), hist, color='steelblue', alpha=0.7, edgecolor='black', linewidth=0.5)
ax.axvline(otsu_threshold, color='red', linestyle='--', linewidth=2, label=f'Otsu threshold = {otsu_threshold}')
ax.set_xlabel('Pixel Intensity (0-255)')
ax.set_ylabel('Pixel Count')
ax.set_title('Histogram: Distribution of Pixel Brightness Values')
ax.legend()
ax.grid(True, alpha=0.3)

# Panel 2: Cumulative Distribution (% of pixels brighter than threshold)
ax = axes[1]
ax.plot(range(256), cumsum_inv * 100, color='green', linewidth=2, label='% pixels BRIGHTER than threshold')
ax.fill_between(range(256), 0, cumsum_inv * 100, alpha=0.2, color='green')
ax.axvline(otsu_threshold, color='red', linestyle='--', linewidth=2, label=f'Otsu threshold = {otsu_threshold}')
ax.axhline(cumsum_inv[otsu_threshold] * 100, color='red', linestyle=':', alpha=0.5)
ax.set_xlabel('Threshold Intensity (0-255)')
ax.set_ylabel('Percentage of Pixels (%)')
ax.set_title('Cumulative Distribution: Percentage of Pixels with Brightness > Threshold')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 100])

# Panel 3: Between-Class Variance
ax = axes[2]
ax.plot(range(256), between_class_variance, color='purple', linewidth=2, label='Between-class variance')
ax.axvline(otsu_threshold, color='red', linestyle='--', linewidth=2, label=f'Optimal threshold = {otsu_threshold}')
ax.scatter([otsu_threshold], [max_variance], color='red', s=200, zorder=5, marker='o', label=f'Maximum variance = {max_variance:.2f}')
ax.fill_between(range(256), 0, between_class_variance, alpha=0.2, color='purple')
ax.set_xlabel('Threshold Intensity (0-255)')
ax.set_ylabel('Between-Class Variance')
ax.set_title('Otsu Objective Function: Between-Class Variance for Each Threshold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n✓ Otsu threshold: {otsu_threshold}")
print(f"✓ At threshold {otsu_threshold}: {cumsum_inv[otsu_threshold]*100:.1f}% of pixels are BRIGHTER")

## 5. Compare Original and Thresholded Images

Apply the Otsu threshold to the image and visualize the segmentation result.

In [ ]:
# Apply the Otsu threshold
_, binary_mask = cv2.threshold(red_img, otsu_threshold, 255, cv2.THRESH_BINARY)

# Display comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original image
ax = axes[0]
ax.imshow(red_img, cmap='gray')
ax.set_title(f'Original Red-Enhanced Image')
ax.axis('off')

# Binary mask (after thresholding)
ax = axes[1]
ax.imshow(binary_mask, cmap='gray')
ax.set_title(f'Binary Mask (Otsu threshold = {otsu_threshold})')
ax.axis('off')

plt.tight_layout()
plt.show()

# Statistics
dark_pixels = (binary_mask == 0).sum()
bright_pixels = (binary_mask == 255).sum()
total_pixels = dark_pixels + bright_pixels

print(f"\n=== Segmentation Results ===")
print(f"Dark pixels (< {otsu_threshold}):  {dark_pixels:8d} ({dark_pixels/total_pixels*100:5.1f}%)")
print(f"Bright pixels (≥ {otsu_threshold}): {bright_pixels:8d} ({bright_pixels/total_pixels*100:5.1f}%)")

## Summary: How Otsu Method Works

The Otsu thresholding algorithm automatically selects the optimal threshold by:

1. **Computing a histogram** of all pixel intensities in the image
2. **For each possible threshold value** (0–255):
   - Partition pixels into two classes: dark (< threshold) and bright (≥ threshold)
   - Calculate the **between-class variance**: a measure of how well-separated the two classes are
3. **Selecting the threshold** that **maximizes the between-class variance**

### Key Insight
- Otsu avoids manual tuning by using an objective criterion: maximize separation between dark and bright regions
- This works well when the image has a clear bimodal histogram (two peaks)
- For the red reflex detection, Otsu finds the boundary between the dark background and the bright red blob

### Connection to Your Red Reflex Analysis
In your pipeline:
- The **red-enhanced image** contains mostly dark background with a bright red reflex blob
- **Otsu threshold** automatically finds the boundary between them
- For very thin ellipses, you may need **lower percentile thresholds** (0.5%–1%) instead of Otsu to capture only the brightest core

This explains why your **sweep mode at 0.5%** gives a much thinner ellipse than the Otsu default!